### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [ ]:
# uv add python-dotenv langchain langchain-openai

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )

##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7)  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print("llm 설정 확인 완료", f"{PROVIDER} / {llm.model_name}")

llm 설정 확인 완료 groq / openai/gpt-oss-120b


In [4]:

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 대량의 텍스트 데이터를 사용해 다음에 올 단어를 예측하도록 신경망을 학습시킵니다. 이 과정에서 트랜스포머 구조의 '
 '셀프‑어텐션 메커니즘을 이용해 문맥을 파악하고, 손실 함수를 최소화하도록 파라미터를 조정합니다. 최종적으로는 인간과 유사한 언어 생성 '
 '능력을 갖춘 모델이 완성됩니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [5]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('**Gemini 모델의 학습 원리 (3문장)**  \n'
 '1. Gemini는 대규모 텍스트와 멀티모달 데이터(이미지·음성·동영상 등)를 활용해 사전 학습(pre‑training)을 진행하며, '
 'Transformer 기반의 디코더와 인코더 구조를 동시에 학습합니다.  \n'
 '2. 사전 학습 단계에서는 마스크드 언어 모델링, 다음 문장 예측, 이미지‑텍스트 정합성 학습 등 다양한 자기 '
 '지도(self‑supervised) 목표를 결합해 일반적인 언어·시각 이해 능력을 획득합니다.  \n'
 '3. 이후 도메인‑특화 데이터와 인간 피드백(RLHF)을 이용한 미세조정(fine‑tuning) 단계에서 사용자의 의도와 안전성을 반영한 '
 '고품질 응답을 생성하도록 최적화됩니다.  \n'
 '\n'
 '---\n'
 '\n'
 '### Gemini 모델의 주요 장점 요약  \n'
 '\n'
 '| 장점 | 설명 |\n'
 '|------|------|\n'
 '| **멀티모달 통합** | 텍스트·이미지·음성 등을 동시에 이해·생성해 복합적인 질문에 대응 가능 |\n'
 '| **대규모 사전 학습** | 수천억 파라미터와 방대한 데이터셋으로 높은 언어 이해·생성 능력 확보 |\n'
 '| **안전·윤리 강화** | 인간 피드백 기반 RLHF와 정교한 필터링으로 유해·편향 콘텐츠 감소 |\n'
 '| **실시간 적응** | 사용자 피드백을 지속적으로 반영해 응답 품질을 빠르게 개선 |\n'
 '| 

#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [6]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

[0, 4, 16, 36, 64]

In [7]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

[0, 4, 16, 36, 64]

In [8]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 4},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT-4는 대규모 텍스트 데이터를 사용해 다음에 올 단어를 예측하도록 설계된 트랜스포머(Transformer) 아키텍처 기반의 언어 '
 '모델입니다. 학습 과정에서는 입력 문맥에 따라 정답 토큰을 맞추는 확률을 최대화하도록 손실 함수를 최소화하며, 이를 위해 역전파와 '
 '옵티마이저(예: Adam)를 이용해 모델 파라미터를 반복적으로 업데이트합니다. 이렇게 학습된 파라미터는 다양한 언어 패턴과 지식을 '
 '일반화하여, 새로운 문장에 대해 일관된 답변을 생성할 수 있게 합니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini 모델은 대규모 텍스트와 멀티모달 데이터를 사용해 사전 학습(pre‑training)을 진행합니다.  \n'
 '이 과정에서 트랜스포머 기반의 자기주의(self‑attention) 메커니즘을 활용해 문맥을 이해하고, 다양한 패턴을 추출합니다.  \n'
 '그 후, 특정 작업에 맞춘 미세조정(fine‑tuning) 단계에서 인간 피드백(예: RLHF)이나 지도학습 데이터를 적용해 성능을 '
 '최적화합니다.  \n'
 '최종적으로, 여러 도메인과 언어에 걸친 일반화 능력을 갖춘 고성능 언어·비전 모델로 완성됩니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Claude 모델은 대규모 트랜스포머 아키텍처를 기반으로, 인터

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [9]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
print(llm.model_name)
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?', additional_kwargs={}, response_metadata={})]
openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
### ExaOne 모델의 학습 원리  

**1. 데이터 수집 및 전처리**  
- **대규모 텍스트 코퍼스**: 웹 페이지, 뉴스 기사, 책, 위키피디아 등 다양한 출처에서 수집한 텍스트 데이터를 사용합니다.  
- **필터링**: 저작권이 있는 자료, 개인 식별 정보(PII), 폭력·혐오·불법 콘텐츠 등은 사전에 필터링하고 제거합니다.  
- **토크나이징(Tokenizing)**: 텍스트를 의미 단위(토큰)로 나누고, 어휘 사전(vocabulary)을 구축합니다. 일반적으로 BPE(Byte‑Pair Encoding)나 WordPiece와 같은 서브워드 방식을 사용해 어휘 크기를 제한합니다.

**2. 모델 아키텍처**  
- **Transformer 기반**: Exa1​은 기본적으로 “Transformer” 아키텍처를 사용합니다.  
  - **Encoder‑Decoder** 혹은 **Decoder‑only** 형태 중 하나이며, 대부분의 최신 대형 언어 모델은 Decoder‑only 구조(예: GPT 시리즈)로 설계됩니다.  
  - **Self‑Attention** 메커니즘을 통해 입력 시퀀스 내 모든 토큰 간의 관계를 동시에 학습합니다.  
- **스케일**: 모델 파라미터 수가 수십 억(10 B)에

In [10]:
# 체인을 생성하여 호출하기
chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"deepseek"})
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
## DeepSeek 모델의 학습 원리

DeepSeek 시리즈(예: DeepSeek‑Coder, DeepSeek‑V2 등)는 **Transformer 기반의 대규모 언어 모델**입니다.  
다음은 일반적인 학습 흐름과 DeepSeek에 적용된 주요 기술 요소를 단계별로 정리한 내용입니다.

| 단계 | 설명 | DeepSeek에 적용된 특징 |
|------|------|------------------------|
| 1️⃣ **데이터 수집·전처리** | - 인터넷 웹 페이지, 위키피디아, 책, 논문, 코드 저장소 등 다양한 도메인에서 **수조(token) 규모**의 텍스트 데이터를 수집.<br>- 중복, 저품질, 개인정보 등은 필터링하고, 토크나이저(보통 BPE/Byte‑Level BPE)를 이용해 **서브워드 단위**로 토큰화. | - DeepSeek‑V2는 약 **2 트릴리언(token)**에 달하는 다국어·다도메인 데이터(영어, 한국어, 중국어 등)를 사용.<br>- 코드 모델(DeepSeek‑Coder)에서는 GitHub·StackOverflow 등에서 추출한 **수백 GB 규모의 코드**와 주석을 포함. |
| 2️⃣ **사전 학습 (Pre‑training)** | - **자기 회귀(autoregressive) 방식**으로 “다음 토큰 예측” 목표를 학습.<br>- 입력 시퀀스 전체에 **Self‑Attention**을 적용해 각 토큰이 다른 모든 토큰과 상호작용하도록 함.<br>- 대규모 **분산 학습**(수천 개 GPU/TPU)으로 **Mixed‑Precision(FP16/ BF16)** 및 **ZeRO/DeepSpeed** 같은 파라미터 효율화 기법을 사용. | - DeepSeek‑V2는 **100 B 파라미터**(≈ 100 억) 규모이며, **Sparse‑MoE(모듈식 전문가)** 구조를 도입해 일부 레이어에서 전문가(Expert)들을 동적으로 선

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [11]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출 (공통 설정 셀의 llm 사용)
response = llm.invoke(messages)

# 결과 출력
print(response.content)

**독감(인플루엔자) 예방 접종을 맞아야 할지 고민 중이시군요.** 아래에서는 독감 예방 접종이 왜 권장되는지, 누가 특히 맞는 것이 좋은지, 그리고 접종을 고려할 때 주의할 점을 단계별로 정리해 드리겠습니다.

---

## 1. 독감 예방 접종이 왜 중요한가요?

| 이유 | 설명 |
|------|------|
| **감염 위험 감소** | 독감 백신은 해당 시즌에 유행할 것으로 예상되는 주요 바이러스 균주 2~3종에 대한 면역을 제공합니다. 접종 후 2주 정도 지나면 대부분의 사람에게 감염 위험이 40~60% 정도 감소합니다. |
| **중증도 완화** | 백신에 의해 완전히 예방되지 않더라도, 감염 시 증상이 경미해지고 입원·중환자실 입실·사망 위험이 크게 낮아집니다. |
| **집단 면역** | 주변 사람들(특히 어린이·노인·기저질환자)에게 전파되는 것을 차단해 전체 인구의 발병률을 낮춥니다. |
| **경제적·사회적 비용 절감** | 감염으로 인한 결근·입원·치료비가 줄어들어 개인·사회 차원에서 비용 부담이 감소합니다. |

---

## 2. 누가 특히 독감 예방 접종을 권장받나요?

| 대상 | 이유 |
|------|------|
| **고령자 (65세 이상)** | 면역력이 약해 중증 독감·합병증 위험이 높음 |
| **만성질환자** (당뇨, 심혈관질환, 천식·COPD, 간·신장질환 등) | 기저질환이 있으면 독감이 악화될 위험이 큼 |
| **임산부** (특히 2~3분기) | 임신 중 면역 변화와 태아 보호를 위해 권장 |
| **어린이 (6개월~5세)** | 면역 체계가 미성숙하고 전파력이 강함 |
| **보건·의료 종사자** | 환자와 직접 접촉이 많아 전파 위험이 높음 |
| **집단 생활자** (학교, 군대, 요양원 등) | 전파 차단이 사회적 안전망에 중요 |
| **면역억제 치료를 받는 사람** | 면역력이 저하돼 감염 위험이 크게 증가 |

> **일반 건강한 성인**도 독감 예방 접종을 하면 개인적·사회적으로 큰 이득이 있습

#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [12]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

[SystemMessage(content='You are a helpful assistant that translates English to Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results.', additional_kwargs={}, response_metadata={})]
체이닝은 여러 번의 LLM 호출을 순차적으로 수행하는 것을 의미할 수 있습니다. 언어 모델은 종종 비결정적이며 오류를 일으킬 수 있기 때문에, 이전 출력물을 확인하거나 큰 작업을 작은 단계로 나누어 여러 번 호출하면 결과를 개선할 수 있습니다.


##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [13]:
# PromptTemplate을 사용하지 않는 경우

# model
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
**태양계 행성 간략 정리**

| 순서 | 행성 | 평균 거리(태양으로부터) | 지름(≈) | 주요 특징 |
|------|------|------------------------|---------|-----------|
| 1 | **수성 (Mercury)** | 0.39 AU (약 5,800만 km) | 4,880 km | 가장 작은 행성, 매우 얇은 대기, 극심한 온도 차이(−173 °C ~ +427 °C) |
| 2 | **금성 (Venus)** | 0.72 AU (약 1.08억 km) | 12,104 km | 지구와 비슷한 크기·질량, 두꺼운 이산화탄소 대기와 황산 구름, 표면 온도 약 462 °C (온실 효과) |
| 3 | **지구 (Earth)** | 1 AU (약 1.5억 km) | 12,742 km | 물이 액체 상태로 존재하는 유일한 행성, 대기 중 산소·질소 비율, 생명체 존재 |
| 4 | **화성 (Mars)** | 1.52 AU (약 2.28억 km) | 6,779 km | 얇은 이산화탄소 대기, 적색 토양(산화철), 극지에 물 얼음, 과거 물 흐름 흔적 |
| 5 | **목성 (Jupiter)** | 5.20 AU (약 7.78억 km) | 139,822 km | 태양계 최대 행성, 가스(수소·헬륨) 행성, 거대한 폭풍(대적점), 79개 위성(가니메데 등) |
| 6 | **토성 (Saturn)** | 9.58 AU (약 1.43억 km) | 116,464 km | 눈에 띄는 고리 시스템(얼음·암석 파편), 가스 행성, 83개 위성(타이탄 등) |
| 7 | **천왕성 (Uranus)** | 19.2 AU (약 2.87억 km) | 50,724 km | 청록색(메탄 대기), 자전축이 98° 기울어 “옆으로 누운” 행성, 27개의 위성 |
| 8 | **해왕성 (Neptune)** | 30.1 AU (약 4.50억

##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [14]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예시는 시스템 프롬프트(초등학생 눈높이)와 같은 톤·같은 형식(제목 + 비유 글머리표 + 한 줄 정리)으로 통일
examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.
- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.
- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.
 한 줄 정리: 힘은 항상 짝으로 움직여요!"""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.
- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.
- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.
 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!"""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

openai/gpt-oss-120b
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': '뉴턴의 운동 법칙을 요약해 주세요.', 'output': '### 뉴턴의 운동 법칙\n- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.\n- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.\n- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.\n 한 줄 정리: 힘은 항상 짝으로 움직여요!'}, {'input': '지구의 대기 구성 요소를 알려주세요.', 'output': '### 지구 대기의 구성\n- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.\n- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.\n- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.\n 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptT

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [15]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    # 함수 자체를 전달 → format() 호출 시점마다 현재 계절을 계산
    partial_variables={"season": get_current_season}  # 동적으로 계절 값 할당
)


# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
llm = get_llm(temperature=0.7)  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print(llm.model_name)
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

openai/gpt-oss-120b
 프롬프트: 가을에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         가을에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: 가을에 나타나는 대표적인 **지구과학 현상**은 기후·대기·수문·지형 과정이 복합적으로 작용하면서 나타나는 현상들입니다.  
아래는 가을에 특히 **자주 관측되거나 활발히 진행되는 3가지 주요 현상**이며, 각각 간단히 설명을 덧붙였습니다.

| # | 현상 | 주요 특징 및 발생 메커니즘 |
|---|------|----------------------------|
| 1 | **태풍(열대 저기압)의 발생·접근·상륙** | - 서북태평양에서는 8월 말부터 10월까지 태풍 활동이 절정에 달합니다.<br>- 가을이 되면 남쪽 해수면 온도가 아직 높아 태풍이 충분히 강해진 뒤, 북쪽으로 이동하면서 한반도·동북아에 상륙하거나 큰 비·강풍을 동반합니다.<br>- 태풍은 강수량 급증, 홍수·산사태 위험, 해안 파도·풍해 등 다양한 지구과학적 영향을 끼칩니다. |
| 2 | **전선·외란성 저기압(엑스트라트로픽 사이클론)의 활발한 활동** | - 가을에는 남북 온도 차이가 크게 벌어지면서 **중위도 전선**이 활발해집니다.<br>- 전선이 만나면서 발생하는 외란성 저기압은 강수·폭풍·강풍을 동반하고, 특히 **‘가을 강우’**라 불리는 장마 이후의 폭우를 유발합니다.<br>- 이때 발생하는 **대기압 골짜기**, **제트기류 남하** 등은 항공·해운·에너지 수급에 큰 영향을 미칩니다. |
| 3 | **가을 안개·역전 현상 및 급격한 온도 변화** | - 가을 저녁·새벽에 대지 표면이 급격히 냉각되면서 **기온 역전**이 자주 발생합니다.<br>- 차가운 공기가 지표면에 머물면서 **안개**가 형성되고, 시야가 급격히 감소합니다.<br>- 역전층 위에서는 대기가 상대적으로 안정돼 **오염 물질·연무**가 축적되기도 하며, 이는 대기질·기상 관측에 중요한 변수가 됩니다. |
|   

In [16]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
season_name = get_current_season()
#season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

현재 계절: 가을


In [17]:

# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain1의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

openai/gpt-oss-120b

 가을에 발생하는 자연 현상:
### 가을에 주로 나타나는 대표적인 지구과학 현상 3가지  

| 현상 | 간단한 설명 |
|------|-------------|
| **1. 낙엽 변색·낙엽 (Leaf senescence & fall)** | 가을이 되면 일조량 감소와 낮·밤 온도 차이가 커지면서 식물의 광합성 효율이 떨어집니다. 엽록소가 분해되고 카로티노이드·안토시아닌 같은 색소가 드러나면서 잎이 노랗거나 붉게 변하고, 결국 잎이 떨어집니다. 이는 탄소·질소 순환에 중요한 역할을 하며, 토양 유기물 형성의 시작점이 됩니다. |
| **2. 대기 역전 현상 (Temperature inversion)** | 가을밤에 땅이 빠르게 식으면서 지표면 근처의 공기가 차가워지고, 위쪽에 있는 따뜻한 공기층이 그 위에 얹히는 **역전층**이 형성됩니다. 역전층은 수직 대류를 억제해 안개·연무가 자주 발생하고, 오염 물질이 하층에 머물러 대기질이 악화되는 원인이 됩니다. 특히 산간·계곡 지역에서 흔히 관찰됩니다. |
| **3. 가을 전선·강수 활동 (Frontal systems & autumn rains)** | 가을에는 남쪽에서 따뜻하고 습한 공기가 북쪽의 차가운 고기압과 마주하면서 **한랭 전선**이 자주 형성됩니다. 전선이 통과할 때 급격한 기압·온도 변화와 함께 구름이 발달하고, 강수(소나기·전선성 비)가 집중적으로 내립니다. 이때 강수량이 많아지면서 토양 수분이 보충되고, 하천 유역의 수위가 상승해 수문학적 변화를 일으킵니다. |

> **요점** – 가을은 식물·대기·수문·기상 시스템이 서로 얽히면서 **잎의 색 변화·대기 역전·전선성 강수**라는 세 가지 주요 지구과학 현상이 뚜렷하게 나타나는 계절입니다. 각각은 생태계, 대기 질, 수자원에 중요한 영향을 미칩니다.


#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [18]:
import requests
from datetime import datetime
from langchain_core.prompts import PromptTemplate

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/USD"


def get_exchange_rate() -> str:
    """USD→KRW 환율과 API 기준 일자를 문자열로 반환합니다."""
    response = requests.get(EXCHANGE_API_URL, timeout=10)
    response.raise_for_status()
    data = response.json()
    base_date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    return f"{base_date} 기준 1달러 = {data['rates']['KRW']}원"


# Partial Prompt 활용 (함수 자체를 전달 → format() 호출 시점마다 실행)
prompt = PromptTemplate(
    template=(
        "오늘 날짜: {today}\n"
        "환율 정보: {info}\n\n"
        "위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. "
        "제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요."
    ),
    input_variables=[],  # 사용자 입력 없음
    partial_variables={
        "today": lambda: datetime.now().strftime("%Y-%m-%d"),
        "info": get_exchange_rate,  # API에서 가져온 데이터 자동 반영
    },
)

# LLM 모델명
print(llm.model_name)

# 프롬프트를 한 번만 생성해 재사용 (API 중복 호출 방지)
formatted = prompt.format()

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(formatted)

# 결과 출력
print(" 프롬프트:", formatted)
print(" 모델 응답:", response.content)

openai/gpt-oss-120b
 프롬프트: 오늘 날짜: 2026-10-07
환율 정보: 2026-10-07 기준 1달러 = 1338.92원

위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. 제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요.
 모델 응답: **환율 상황 간단 분석 (2026‑10‑07 기준)**  

- **현재 환율**: 1 USD = 1,338.92 KRW  
- **의미**: 1달러를 사려면 약 1,339원의 원화가 필요합니다. 이는 원화가 달러 대비 약 1,300원 정도의 가치를 가지고 있다는 것을 보여줍니다.  
- **제한점**: 과거 환율 데이터나 외부 요인(경제 지표, 정책, 뉴스 등)이 제공되지 않았기 때문에, 이번 환율이 이전에 비해 상승했는지 하락했는지, 혹은 평균 수준인지 판단할 수 없습니다.  

**요약**  
현재 시점에서 달러당 1,338.92원의 원화가 요구되는 상황이며, 추가적인 비교 자료가 없으므로 이 수치 자체만으로는 환율이 “높다”·“낮다” 등을 평가하기 어렵습니다. 필요하시면 과거 데이터나 시장 동향을 함께 제공해 주시면 보다 구체적인 분석이 가능합니다.
